# 00 Inspect Data

**Purpose.** Inspect the raw ion-list files: scale, mass range, centroiding, intensity floor.

**Thesis reference.** §4.2 Data Characterisation

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [ ]:
DATA_ROOT = "../ahen_data/filtered_tsv_2"       # folder containing the TSVs
LABELS_FILE = "training_data__2_.json"          # label file
FILE_PREFIX = "WCmsdata_s"                      # stripped to get patient ID
N_SAMPLE_FILES = 5                              # files to profile in detail

## Code

In [8]:
"""
=============
Verifies:
  1. TSV format  --- delimiter, header presence, column order, dtypes
  2. File count  --- how many patients actually have data
  3. Missing-data bias --- whether exclusion correlates with CKD status
  4. Observed ranges --- m/z, retention time, ions per file
  5. Sparsity    --- whether 0.02 Da binning is appropriate

Chapter 4
"""

import json
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

def rule(title):
    print("\n" + "=" * 68)
    print(title)
    print("=" * 68)

# 1. Format
def inspect_format(path):
    rule("1. FILE FORMAT")
    print(f"File: {path.name}\n")

    with open(path, "r", errors="replace") as fh:
        head = [next(fh, "") for _ in range(3)]
    print("First 3 raw lines:")
    for i, line in enumerate(head, 1):
        print(f"  {i}: {line.rstrip()[:100]}")

    first = head[0].rstrip()
    for name, ch in [("tab", "\t"), ("comma", ","), ("semicolon", ";"),
                     ("space", " ")]:
        n = first.count(ch)
        if n:
            print(f"\n  delimiter '{name}' -> {n + 1} fields")

    fields = first.split("\t")
    numeric = True
    for f in fields:
        try:
            float(f)
        except ValueError:
            numeric = False
    print(f"\n  Header row present: {'NO' if numeric else 'YES -- add skiprows=1 to the loaders'}")
    print(f"  Field count: {len(fields)}")

    df = pd.read_csv(path, sep="\t", nrows=5000, header=None)
    print(f"\n  Column dtypes (first 5000 rows): "
          f"{[str(t) for t in df.dtypes]}")
    print("\n  Column ranges:")
    guess = {}
    for c in df.columns:
        col = pd.to_numeric(df[c], errors="coerce").dropna()
        if not len(col):
            continue
        lo, hi = col.min(), col.max()
        print(f"    col {c}: [{lo:.4f}, {hi:.4f}]")
        if 0 <= lo and hi <= 200:
            guess[c] = "time"
        elif 40 <= lo and hi <= 3000:
            guess[c] = "mz"
        else:
            guess[c] = "intensity"
    print(f"\n  Inferred order: {[guess.get(c, '?') for c in df.columns]}")
    print("  Expected by the pipeline: ['time', 'mz', 'intensity']")
    if [guess.get(c) for c in df.columns] != ["time", "mz", "intensity"]:
        print("\n  >>> MISMATCH.")
    else:
        print("  >>> MATCH.")

# 2 & 3. Coverage and missing-data bias
def inspect_coverage(files, labels):
    rule("2. COHORT COVERAGE")
    have = {f.stem.replace(FILE_PREFIX, "") for f in files}
    labelled = set(labels)

    print(f"  TSV files on disk:        {len(files)}")
    print(f"  Labelled patients:        {len(labelled)}")
    print(f"  Both label and data:      {len(have & labelled)}")
    print(f"  Data but no label:        {len(have - labelled)}")
    print(f"  Label but no data:        {len(labelled - have)}")

    usable = sorted(have & labelled)
    y = np.array([labels[p] for p in usable])
    print(f"\n  ANALYSED COHORT: n = {len(y)}  "
          f"({int(y.sum())} CKD+ / {int((1 - y).sum())} CKD-)  "
          f"positive rate {y.mean():.1%}")

    rule("3. MISSING-DATA BIAS")
    missing = sorted(labelled - have)
    if not missing:
        print("  No labelled patients lack data. Nothing to test.")
        return usable

    cp = Counter(labels[p] for p in usable)
    cm = Counter(labels[p] for p in missing)
    n_p, n_m = len(usable), len(missing)

    print(f"  WITH data    n={n_p:3d}   CKD+ {cp[1]:3d} "
          f"({cp[1] / n_p:.1%})   CKD- {cp[0]:3d}")
    print(f"  WITHOUT data n={n_m:3d}   CKD+ {cm[1]:3d} "
          f"({cm[1] / n_m:.1%})   CKD- {cm[0]:3d}")

    try:
        from scipy.stats import fisher_exact
        odds, p = fisher_exact([[cp[1], cp[0]], [cm[1], cm[0]]])
        print(f"\n  Fisher exact: odds ratio {odds:.3f}, p = {p:.5f}")
        if p < 0.05:
            print("\n  >>> Exclusion is NOT random with respect to CKD")
        else:
            print("\n  >>> No significant association.")
    except ImportError:
        print("  (scipy not available -- skipping significance test)")

    return usable

# 4 & 5. Content and sparsity
def inspect_content(files, n_sample):
    rule("4. DATA CONTENT")
    sample = files[:n_sample]
    stats = []
    for f in sample:
        df = pd.read_csv(f, sep="\t", names=["time", "mz", "intensity"],
                         dtype=str, low_memory=False)
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        bad = df.isna().any(axis=1).sum()
        df = df.dropna()
        stats.append({
            "pid": f.stem.replace(FILE_PREFIX, ""),
            "rows": len(df),
            "bad": int(bad),
            "mz_lo": df["mz"].min(), "mz_hi": df["mz"].max(),
            "rt_lo": df["time"].min(), "rt_hi": df["time"].max(),
            "int_med": df["intensity"].median(),
            "int_hi": df["intensity"].max(),
            "df": df,
        })
        s = stats[-1]
        print(f"  {s['pid']}: {s['rows']:>9,} rows "
              f"({s['bad']} unparseable) | "
              f"m/z [{s['mz_lo']:.1f}, {s['mz_hi']:.1f}] | "
              f"RT [{s['rt_lo']:.2f}, {s['rt_hi']:.2f}]")

    print(f"\n  Median rows per file: "
          f"{int(np.median([s['rows'] for s in stats])):,}")
    print(f"  Overall m/z range: "
          f"[{min(s['mz_lo'] for s in stats):.1f}, "
          f"{max(s['mz_hi'] for s in stats):.1f}]")
    print(f"  Overall RT range:  "
          f"[{min(s['rt_lo'] for s in stats):.2f}, "
          f"{max(s['rt_hi'] for s in stats):.2f}]")
    print("\n  Pipeline assumes m/z [50, 1650] and RT [0, 37].")
    print("  >>> If the observed ranges differ, update MZ_MIN/MZ_MAX and RT_MIN/RT_MAX")

    rule("5. SPARSITY AT 0.02 Da")
    for s in stats[:3]:
        mz = s["df"]["mz"].to_numpy()
        it = s["df"]["intensity"].to_numpy()
        for res in (0.02, 0.05, 0.10):
            edges = np.arange(50.0, 1650.0 + res, res)
            h, _ = np.histogram(mz, bins=edges, weights=it)
            occ = (h > 0).sum() / len(h)
            print(f"  {s['pid']}  {res:.2f} Da -> {len(h):>7,} bins, "
                  f"{occ:6.2%} occupied")
        print()
    print("  Rule of thumb: < ~2% occupancy - very sparse.")

def main():
    root = Path(DATA_ROOT)
    if not root.is_dir():
        print(f"ERROR: DATA_ROOT not found: {root.resolve()}")
        print("Edit the CONFIG block at the top of this file.")
        return

    files = sorted(root.glob("*.tsv"))
    if not files:
        print(f"ERROR: no .tsv files in {root.resolve()}")
        return

    inspect_format(files[0])

    if os.path.exists(LABELS_FILE):
        with open(LABELS_FILE) as fh:
            obj = json.load(fh)
        labels = obj.get("labels", obj)
        inspect_coverage(files, labels)
    else:
        print(f"\n[skip] labels file not found: {LABELS_FILE}")

    inspect_content(files, N_SAMPLE_FILES)

    rule("DONE")

## Run

In [9]:
main()


1. FILE FORMAT
File: WCmsdata_sP0001.tsv

First 3 raw lines:
  1: time	mz	intensity
  2: 0.024	50.37	1.013
  3: 0.024	50.42	1.013

  delimiter 'tab' -> 3 fields

  Header row present: YES -- add skiprows=1 to the loaders
  Field count: 3

  Column dtypes (first 5000 rows): ['object', 'object', 'object']

  Column ranges:
    col 0: [0.0240, 0.0320]
    col 1: [50.1600, 1649.4600]
    col 2: [1.0130, 64.4600]

  Inferred order: ['time', 'mz', 'time']
  Expected by the pipeline: ['time', 'mz', 'intensity']

  >>> MISMATCH.

[skip] labels file not found: training_data__2_.json

4. DATA CONTENT
  P0001: 28,132,601 rows (1 unparseable) | m/z [50.0, 1650.9] | RT [0.02, 37.00]
  P0002: 28,153,190 rows (1 unparseable) | m/z [49.9, 1651.1] | RT [0.02, 37.00]
  P0004: 28,470,787 rows (1 unparseable) | m/z [50.0, 1651.0] | RT [0.02, 37.00]
  P0005: 28,126,931 rows (1 unparseable) | m/z [50.0, 1651.0] | RT [0.02, 37.00]
  P0006: 28,565,262 rows (1 unparseable) | m/z [50.0, 1651.0] | RT [0.02, 37.

In [10]:
# H. Re-derive the mass dispersion figure
import numpy as np
import pandas as pd
from pathlib import Path

DATA_ROOT  = "../ahen_data/filtered_tsv_2"
N_PATIENTS = 40
TOP_PEAKS  = 300         # strongest peaks per patient
TOL        = 0.30        # Da: how far apart two peaks can be and still be treated as the same nominal mass

files = sorted(Path(DATA_ROOT).glob("*.tsv"))[:N_PATIENTS]
print(f"reading {len(files)} files\n")

peaks = []
for k, f in enumerate(files, 1):
    df = pd.read_csv(f, sep="\t", usecols=["mz", "intensity"])
    top = df.nlargest(TOP_PEAKS, "intensity")
    peaks.append(top["mz"].to_numpy())
    print(f"  {k}/{len(files)}", end="\r", flush=True)
print(" " * 20)

allmz = np.sort(np.concatenate(peaks))

# cluster peaks that lie within TOL of one another
clusters, cur = [], [allmz[0]]
for v in allmz[1:]:
    if v - cur[-1] <= TOL:
        cur.append(v)
    else:
        clusters.append(cur); cur = [v]
clusters.append(cur)

# keep clusters seen in a decent share of patients
good = [np.array(c) for c in clusters if len(c) >= 0.5 * len(files)]
print(f"{len(good)} recurrent masses seen in at least half the patients\n")

if not good:
    raise SystemExit("no recurrent masses --- try raising TOL or TOP_PEAKS")

sds     = np.array([c.std(ddof=1) for c in good])
ranges  = np.array([c.max() - c.min() for c in good])
centres = np.array([c.mean() for c in good])

print("spread of a single recurrent mass across patients")
print(f"  median standard deviation : {np.median(sds):.4f} Da")
print(f"  median full range         : {np.median(ranges):.4f} Da")
print(f"  90th percentile of range  : {np.percentile(ranges, 90):.4f} Da")
print(f"\n  half-range (the +/- figure): {np.median(ranges)/2:.4f} Da")

ppm = 1e6 * np.median(sds) / np.median(centres)
print(f"\n  expressed in relative terms: {ppm:.0f} ppm at m/z {np.median(centres):.0f}")

bin_w = 0.02
print(f"\nprofile bin width in use    : {bin_w} Da")
print(f"median spread                : {np.median(ranges):.4f} Da")
if np.median(ranges) <= bin_w * 1.5:
    print("  -> a recurrent mass mostly falls in one bin; the bin width is")
    print("     appropriate to the measured accuracy.")
else:
    print("  -> a recurrent mass spans more than one bin; consider a wider bin.")

reading 40 files

                    
21 recurrent masses seen in at least half the patients

spread of a single recurrent mass across patients
  median standard deviation : 0.0773 Da
  median full range         : 0.5100 Da
  90th percentile of range  : 1.8200 Da

  half-range (the +/- figure): 0.2550 Da

  expressed in relative terms: 148 ppm at m/z 522

profile bin width in use    : 0.02 Da
median spread                : 0.5100 Da
  -> a recurrent mass spans more than one bin; consider a wider bin.


In [11]:
import json
lab_all = json.load(open("labels/labels_detect_any_ckd.json"))["labels"]   # 1 = any CKD
lab_sev = json.load(open("labels/labels_severity_all.json"))["labels"]     # 1 = mCKD

# group 2 (moderate) = 1, everything else including healthy = 0
out = {p: (1 if lab_sev.get(p) == 1 else 0) for p in lab_all}
json.dump({"task": "progression_vs_baseline", "labels": out},
          open("labels/labels_progression.json", "w"), indent=1)
print(len(out), "patients,", sum(out.values()), "moderate")

877 patients, 194 moderate
